In [6]:
!pip install opencv-python-headless matplotlib ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.1 MB/s eta 0:00:00


In [7]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider
from IPython.display import display


In [8]:
class Region:
    def __init__(self, x, y, w, h, image):
        self.x, self.y, self.w, self.h = x, y, w, h
        patch = image[y:y+h, x:x+w]
        self.n = w * h
        self.sum = float(np.sum(patch))
        self.sum_sq = float(np.sum(patch.astype(np.float64)**2))
        self.update_stats()

    def update_stats(self):
        self.mean = self.sum / self.n
        var = (self.sum_sq - (self.sum**2)/self.n) / self.n
        self.var = max(0.0, var)
        self.std = np.sqrt(self.var)

def are_adjacent(r1, r2):
    vert = ((r1.x + r1.w == r2.x or r2.x + r2.w == r1.x)
            and not (r1.y + r1.h <= r2.y or r2.y + r2.h <= r1.y))
    horz = ((r1.y + r1.h == r2.y or r2.y + r2.h == r1.y)
            and not (r1.x + r1.w <= r2.x or r2.x + r2.w <= r1.x))
    return vert or horz


In [9]:
def split_pass(region, split_threshold, min_size, image):
    if region.std <= split_threshold or region.w <= min_size or region.h <= min_size:
        return [region]
    hw, hh = region.w // 2, region.h // 2
    quads = [
        (region.x, region.y, hw, hh),
        (region.x+hw, region.y, region.w-hw, hh),
        (region.x, region.y+hh, hw, region.h-hh),
        (region.x+hw, region.y+hh, region.w-hw, region.h-hh),
    ]
    leaves = []
    for x, y, w, h in quads:
        if w > 0 and h > 0:
            leaves.extend(split_pass(Region(x, y, w, h, image), split_threshold, min_size, image))
    return leaves

def build_adjacency(regions):
    adj = [[] for _ in regions]
    for i in range(len(regions)):
        for j in range(i+1, len(regions)):
            if are_adjacent(regions[i], regions[j]):
                adj[i].append(j)
                adj[j].append(i)
    return adj

def merge_pass(regions, merge_threshold):
    parent = list(range(len(regions)))
    stats = [{'n': r.n, 'sum': r.sum, 'sum_sq': r.sum_sq} for r in regions]
    adj = build_adjacency(regions)

    def find(u):
        if parent[u] != u:
            parent[u] = find(parent[u])
        return parent[u]

    def union(u, v):
        ru, rv = find(u), find(v)
        if ru == rv:
            return False
        if ru > rv:
            ru, rv = rv, ru
        parent[rv] = ru
        s1, s2 = stats[ru], stats[rv]
        s1['n'] += s2['n']
        s1['sum'] += s2['sum']
        s1['sum_sq'] += s2['sum_sq']
        return True

    changed = True
    while changed:
        changed = False
        for i, nbrs in enumerate(adj):
            for j in nbrs:
                if i >= j:
                    continue
                ri, rj = find(i), find(j)
                if ri != rj:
                    s1, s2 = stats[ri], stats[rj]
                    n = s1['n'] + s2['n']
                    var = (s1['sum_sq'] + s2['sum_sq'] - (s1['sum'] + s2['sum'])**2 / n) / n
                    if np.sqrt(max(0, var)) <= merge_threshold:
                        if union(ri, rj):
                            changed = True

    groups = {}
    for idx, r in enumerate(regions):
        root = find(idx)
        groups.setdefault(root, []).append(r)
    return list(groups.values())

def visualize_segments(groups, shape):
    seg = np.zeros((shape[0], shape[1], 3), dtype=np.uint8)
    for group in groups:
        color = np.random.randint(50, 255, 3)
        for reg in group:
            seg[reg.y:reg.y+reg.h, reg.x:reg.x+reg.w] = color
    return seg


In [16]:
def run_segmentation(gray_img, split_th, merge_th, min_sz):
    root = Region(0, 0, gray_img.shape[1], gray_img.shape[0], gray_img)
    leaves = split_pass(root, split_th, min_sz, gray_img)
    groups = merge_pass(leaves, merge_th)
    seg_img = visualize_segments(groups, gray_img.shape)
    return seg_img, len(groups)


In [11]:
# Load image
ing = cv2.imread("input.jpg")
if ing is None:
    raise FileNotFoundError("input.jpg not found in current directory")
gray = cv2.cvtColor(ing, cv2.COLOR_BGR2GRAY)


In [17]:
def interactive_segmentation(split_th, merge_th, min_sz, kernel_sz, sigma):
    if kernel_sz % 2 == 0:
        kernel_sz += 1  # ensure it's odd
    blurred = cv2.GaussianBlur(gray, (kernel_sz, kernel_sz), sigma)
    seg_img, seg_count = run_segmentation(blurred, split_th, merge_th, min_sz)

    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(seg_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Split={split_th}, Merge={merge_th}, Min={min_sz}, k={kernel_sz}, σ={sigma}\nSegments: {seg_count}")
    plt.axis('off')
    plt.show()


In [18]:
interact(
    interactive_segmentation,
    split_th=IntSlider(min=2, max=40, step=1, value=16, description='Split Th'),
    merge_th=IntSlider(min=2, max=40, step=1, value=16, description='Merge Th'),
    min_sz=IntSlider(min=4, max=64, step=1, value=16, description='Min Size'),
    kernel_sz=IntSlider(min=1, max=21, step=2, value=5, description='Kernel Size'),
    sigma=FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='Sigma'),
)


interactive(children=(IntSlider(value=16, description='Split Th', max=40, min=2), IntSlider(value=16, descript…

<function __main__.interactive_segmentation(split_th, merge_th, min_sz, kernel_sz, sigma)>

In [ ]:
# 20 18 22 7 1.6

In [19]:
import datetime

def save_segmentation_image(split_th, merge_th, min_sz, kernel_sz, sigma, filename=None):
    if kernel_sz % 2 == 0:
        kernel_sz += 1
    blurred = cv2.GaussianBlur(gray, (kernel_sz, kernel_sz), sigma)
    seg_img, seg_count = run_segmentation(blurred, split_th, merge_th, min_sz)

    # Varsayılan dosya ismi
    if filename is None:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"seg_output_st{split_th}_mt{merge_th}_ms{min_sz}_k{kernel_sz}_s{sigma:.1f}_{timestamp}.jpg"

    output_path = os.path.join("outputs", filename)
    os.makedirs("outputs", exist_ok=True)
    cv2.imwrite(output_path, seg_img)
    print(f"✅ Saved segmentation to '{output_path}' with {seg_count} segments.")
    return output_path


In [22]:
save_segmentation_image(split_th=20, merge_th=18, min_sz=22, kernel_sz=7, sigma=1.6, filename="my_final_segmentation.jpg")


✅ Saved segmentation to 'outputs/my_final_segmentation.jpg' with 1632 segments.


'outputs/my_final_segmentation.jpg'

In [23]:
from google.colab import files
files.download("outputs/my_final_segmentation.jpg")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>